# Point 7: k-NN sensitivity with a train-only graph (15 and 25 m/s)

Runs `run_knn_sensitivity_fixed` for k ∈ {1, 3, 5, 7, 10, 15} on the 39-episode population in both regimes, then the extended analysis: k-NN branch alone, similarity of the k-NN descriptor across k, and the importance of the k-NN block in the meta-learner. The E5.5 near-duplicate audit is in `01_protocol_vs_model_15ms.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys, importlib
shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_fixed.py', '/content/pcmef_fixed.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import pcmef_fixed as P
importlib.reload(P)

In [ ]:
import torch
print('CUDA aktif:', torch.cuda.is_available())

In [ ]:
import pandas as pd
P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'

def load_and_filter(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (df['Speed'].round(2).astype(str) + '_' +
                           df['Scenario'].astype(str) + '_' +
                           (df['Time'] // 60).astype(int).astype(str))
    speed_range = {15: (13, 17), 25: (23, 27)}
    lo, hi = speed_range[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

dfs15 = load_and_filter(P15, 15)
dfs25 = load_and_filter(P25, 25)

In [ ]:
import os
import json

# Restrict to the 39-episode population used by RF-Stat/LSTM/BERT
dfs15, feasible_15, n_dropped_15 = P.filter_feasible_episodes(dfs15, min_duration_sec=15)
dfs25, feasible_25, n_dropped_25 = P.filter_feasible_episodes(dfs25, min_duration_sec=15)
print(f"15m/s dibuang: {n_dropped_15} episode | 25m/s dibuang: {n_dropped_25} episode")

OUT_KNN = '/content/drive/MyDrive/Detector_Results/knn_sensitivity'
os.makedirs(OUT_KNN, exist_ok=True)
# Set True to delete cached k-NN results and recompute everything.
CLEAR_KNN_CACHE = False
if CLEAR_KNN_CACHE:
    for f in os.listdir(OUT_KNN):
        if f.startswith('KNN_15ms') or f.startswith('KNN_25ms'):
            os.remove(os.path.join(OUT_KNN, f))

def run_knn_full_safe(df_speed, target_speed, out_dir, seeds=None, **kw):
    seeds = seeds or P.SEEDS
    results = []
    for seed in seeds:
        fp = os.path.join(out_dir, f'KNN_{target_speed}ms_seed{seed}.json')
        if os.path.exists(fp):
            results.append(json.load(open(fp)))
            continue
        try:
            cache = P.build_embeddings_cache(df_speed, seed, **kw)
            if cache is None:
                print(f'  [seed {seed}] dilewati (data tidak cukup)')
                continue
            r = P.run_knn_sensitivity_fixed(df_speed, seed, embeddings_cache=cache)
        except ValueError as e:
            print(f'  [seed {seed}] DILEWATI -- kelas hilang setelah align_multi ({e})')
            continue
        r['seed'] = int(seed)
        r['target_speed'] = target_speed
        json.dump(r, open(fp, 'w'), indent=2)
        results.append(r)
        print(f'  [seed {seed:4d}] selesai -> {fp}')
    return results

knn_res15 = run_knn_full_safe(dfs15, 15, OUT_KNN)
knn_res25 = run_knn_full_safe(dfs25, 25, OUT_KNN)

print("=== kNN sensitivity @ 15 m/s ===")
P.aggregate_knn(knn_res15)

print("\n=== kNN sensitivity @ 25 m/s ===")
P.aggregate_knn(knn_res25)

## Extended analysis: run (resumable)

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

def run_knn_sensitivity_extended(df_speed, seed, embeddings_cache=None,
                                   k_values=(1, 3, 5, 7, 10, 15), step_sec=1):
    c = embeddings_cache
    if c is None:
        raise ValueError('butuh embeddings_cache')
    WINDOW_SIZES = P.WINDOW_SIZES

    def assemble(split, sp_arr):
        arrs = [(c['keys'][ws][split], c['emb'][ws][split]) for ws in WINDOW_SIZES]
        block_widths = [c['emb'][ws][split].shape[1] for ws in WINDOW_SIZES]
        block_names = [f'emb_{ws}s' for ws in WINDOW_SIZES]
        arrs.append((c['keys'][15][split], c['xgb_proba'][split]))
        block_widths.append(c['xgb_proba'][split].shape[1]); block_names.append('xgb_proba')
        if sp_arr is not None:
            arrs.append((c['keys'][15][split], sp_arr))
            block_widths.append(sp_arr.shape[1]); block_names.append('knn_descriptor')
        aligned, keep, _ = P.align_multi(c['keys'][15][split], arrs)
        return np.hstack(aligned), c['y'][split][keep], block_widths, block_names

    def fit_meta(Zva, yva, Zte, yte, min_val_for_internal_split=60):
        base_params = dict(objective='multi:softprob', num_class=3,
                           n_estimators=300, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           min_child_weight=3, gamma=0.1, reg_alpha=0.1,
                           reg_lambda=1.0, random_state=seed, eval_metric='mlogloss')
        classes_va = np.unique(yva)
        can_split_internal = (len(yva) >= min_val_for_internal_split
                              and len(classes_va) >= 2
                              and min(np.bincount(yva.astype(int))[cc]
                                     for cc in classes_va.astype(int)) >= 4)
        if can_split_internal:
            Zfit, Zes, yfit, yes = train_test_split(
                Zva, yva, test_size=0.25, stratify=yva, random_state=seed)
        else:
            Zfit, yfit, Zes, yes = Zva, yva, None, None
        sc = RobustScaler().fit(Zfit)
        cw = compute_class_weight('balanced', classes=np.unique(yfit), y=yfit)
        sw = np.array([cw[list(np.unique(yfit)).index(v)] for v in yfit])
        if Zes is not None:
            m = XGBClassifier(**base_params, early_stopping_rounds=20)
            m.fit(sc.transform(Zfit), yfit, sample_weight=sw,
                  eval_set=[(sc.transform(Zes), yes)], verbose=False)
        else:
            m = XGBClassifier(**base_params)
            m.fit(sc.transform(Zfit), yfit, sample_weight=sw, verbose=False)
        yp = m.predict(sc.transform(Zte))
        _, _, f1, _ = precision_recall_fscore_support(yte, yp, average='macro', zero_division=0)
        return float(accuracy_score(yte, yp)), float(f1), m.feature_importances_

    out = {}
    Zva, yva, _, _ = assemble('val', None)
    Zte, yte, _, _ = assemble('test', None)
    acc, f1, _ = fit_meta(Zva, yva, Zte, yte)
    out['no_knn'] = dict(accuracy=acc, macro_f1=f1, graph_density=0.0)

    sp_test_by_k = {}
    for k in k_values:
        train_episodes = [key[0] for key in c['keys'][15]['train']]
        knn = P.SpatialKNN(k=k).fit(c['X15']['train'], train_episodes=train_episodes)
        sp = {}
        for s in ['train', 'val', 'test']:
            q_eps = train_episodes if s == 'train' else None
            sp[s] = knn.transform(c['X15'][s], is_train=(s == 'train'), query_episodes=q_eps)
        sp_test_by_k[k] = sp['test']

        Zva, yva, bw, bn = assemble('val', sp['val'])
        Zte, yte, _, _ = assemble('test', sp['test'])
        acc, f1, fi = fit_meta(Zva, yva, Zte, yte)

        bounds = np.cumsum([0] + bw)
        importance_per_block = {name: float(fi[bounds[i]:bounds[i+1]].sum())
                                 for i, name in enumerate(bn)}
        total_imp = float(fi.sum())
        knn_frac = importance_per_block['knn_descriptor'] / total_imp if total_imp > 0 else 0.0

        out[f'k={k}'] = dict(accuracy=acc, macro_f1=f1,
                             graph_density=knn.graph_density(len(c['X15']['test'])),
                             importance_per_block=importance_per_block,
                             knn_importance_frac=knn_frac)

        sc_b = RobustScaler().fit(sp['train'])
        mb = XGBClassifier(objective='multi:softprob', num_class=3,
                           n_estimators=300, max_depth=4, learning_rate=0.05,
                           random_state=seed, eval_metric='mlogloss')
        mb.fit(sc_b.transform(sp['train']), c['y']['train'], verbose=False)
        yp = mb.predict(sc_b.transform(sp['test']))
        _, _, f1b, _ = precision_recall_fscore_support(c['y']['test'], yp, average='macro', zero_division=0)
        out[f'knn_branch_only_k{k}'] = dict(
            accuracy=float(accuracy_score(c['y']['test'], yp)),
            macro_f1=float(f1b), graph_density=out[f'k={k}']['graph_density'])

    descriptor_comparison = {}
    ks_sorted = sorted(sp_test_by_k.keys())
    for i in range(len(ks_sorted)):
        for j in range(i + 1, len(ks_sorted)):
            k1, k2 = ks_sorted[i], ks_sorted[j]
            a, b = sp_test_by_k[k1], sp_test_by_k[k2]
            mean_abs_diff = float(np.mean(np.abs(a - b)))
            corr = float(np.corrcoef(a.ravel(), b.ravel())[0, 1]) if a.size > 1 else float('nan')
            descriptor_comparison[f'{k1}_vs_{k2}'] = dict(mean_abs_diff=mean_abs_diff, correlation=corr)
    out['descriptor_comparison'] = descriptor_comparison
    return out


import os, json
OUT_KNN_EXT = '/content/drive/MyDrive/Detector_Results/knn_sensitivity_ext'
os.makedirs(OUT_KNN_EXT, exist_ok=True)

def run_knn_ext_safe(df_speed, target_speed, out_dir, seeds=None, **kw):
    seeds = seeds or P.SEEDS
    results = []
    for seed in seeds:
        fp = os.path.join(out_dir, f'KNNEXT_{target_speed}ms_seed{seed}.json')
        if os.path.exists(fp):
            results.append(json.load(open(fp))); continue
        try:
            cache = P.build_embeddings_cache(df_speed, seed, **kw)
            if cache is None:
                print(f'  [seed {seed}] dilewati (data tidak cukup)'); continue
            r = run_knn_sensitivity_extended(df_speed, seed, embeddings_cache=cache)
        except ValueError as e:
            print(f'  [seed {seed}] DILEWATI -- {e}'); continue
        r['seed'] = int(seed); r['target_speed'] = target_speed
        json.dump(r, open(fp, 'w'), indent=2)
        results.append(r)
        print(f'  [seed {seed:4d}] selesai -> {fp}')
    return results

knn_ext_res15 = run_knn_ext_safe(dfs15, 15, OUT_KNN_EXT)
knn_ext_res25 = run_knn_ext_safe(dfs25, 25, OUT_KNN_EXT)
print(f"15 m/s: {len(knn_ext_res15)}/{len(P.SEEDS)} seed selesai")
print(f"25 m/s: {len(knn_ext_res25)}/{len(P.SEEDS)} seed selesai")

## Extended analysis: results (20 seeds)

In [ ]:
import json, os
import numpy as np
import pandas as pd

OUT_KNN_EXT = '/content/drive/MyDrive/Detector_Results/knn_sensitivity_ext'
K_VALUES = (1, 3, 5, 7, 10, 15)

def load_all(speed):
    out = []
    for seed in P.SEEDS:
        fp = os.path.join(OUT_KNN_EXT, f'KNNEXT_{speed}ms_seed{seed}.json')
        out.append(json.load(open(fp)))
    return out

res15 = load_all(15)
res25 = load_all(25)

# ======================================================================
# Butir 2: knn_branch_only utk SEMUA k -- apakah cabang kNN sendirian
# punya sinyal prediktif (dibandingkan dgn no_knn & ensemble penuh)?
# ======================================================================
print("=== Butir 2: knn_branch_only per-k (akurasi, 20 seed) ===")
for speed, res in [(15, res15), (25, res25)]:
    print(f"\n-- {speed} m/s --")
    no_knn_acc = np.array([r['no_knn']['accuracy'] for r in res])
    print(f"  no_knn (referensi, tanpa cabang kNN): mean={no_knn_acc.mean()*100:.2f}%  SD={no_knn_acc.std(ddof=1)*100:.2f}%")
    for k in K_VALUES:
        bo_acc = np.array([r[f'knn_branch_only_k{k}']['accuracy'] for r in res])
        full_acc = np.array([r[f'k={k}']['accuracy'] for r in res])
        print(f"  k={k:2d}: branch_only={bo_acc.mean()*100:.2f}%(SD={bo_acc.std(ddof=1)*100:.2f}%)  "
              f"ensemble_penuh={full_acc.mean()*100:.2f}%(SD={full_acc.std(ddof=1)*100:.2f}%)")

# ======================================================================
# Butir 3: deskriptor k-NN beda antar-k? (selisih absolut & korelasi,
# dirata-rata lewat 20 seed)
# ======================================================================
print("\n=== Butir 3: perbedaan deskriptor k-NN antar nilai-k (rata-rata 20 seed) ===")
for speed, res in [(15, res15), (25, res25)]:
    print(f"\n-- {speed} m/s --")
    pairs = list(res[0]['descriptor_comparison'].keys())
    for pair in pairs:
        diffs = np.array([r['descriptor_comparison'][pair]['mean_abs_diff'] for r in res])
        corrs = np.array([r['descriptor_comparison'][pair]['correlation'] for r in res])
        print(f"  {pair:12s}: mean_abs_diff={diffs.mean():.4f}(SD={diffs.std(ddof=1):.4f})  "
              f"correlation={corrs.mean():.4f}(SD={corrs.std(ddof=1):.4f})")

# ======================================================================
# Butir 4: feature importance meta-learner per blok, per-k (rata-rata 20 seed)
# ======================================================================
print("\n=== Butir 4: feature importance meta-learner -- fraksi utk blok knn_descriptor ===")
for speed, res in [(15, res15), (25, res25)]:
    print(f"\n-- {speed} m/s --")
    for k in K_VALUES:
        fracs = np.array([r[f'k={k}']['knn_importance_frac'] for r in res])
        print(f"  k={k:2d}: knn_importance_frac mean={fracs.mean()*100:.2f}%  SD={fracs.std(ddof=1)*100:.2f}%  "
              f"min={fracs.min()*100:.2f}%  max={fracs.max()*100:.2f}%")
    # breakdown semua blok utk k=5 sbg contoh representatif
    print(f"  -- breakdown semua blok @ k=5 (contoh) --")
    blocks = res[0]['k=5']['importance_per_block'].keys()
    for b in blocks:
        vals = np.array([r['k=5']['importance_per_block'][b] for r in res])
        print(f"    {b:16s}: mean={vals.mean():.4f}  SD={vals.std(ddof=1):.4f}")